# Phase 1 — Weak labels from radiology reports (LLM descriptor extraction)

**What:** runs a local instruction LLM over every unique train report, extracts one
closed-vocabulary descriptor per target (12 targets), then deterministically maps
descriptors → calibrated 0–1 scores using isotonic regression fit on the 58 gold studies.

**How to run:**
1. Set `MODE = "probe"` first. Attach the competition data. **GPU accelerator ON, internet ON.**
2. Probe = 58 gold reports + 42 others. Gates: macro AUC ≥ 0.86 vs gold, parse-fail < 15%,
   projected full run < 8h. If any gate fails, the notebook stops — do not run full mode.
3. Only after probe PASS: set `MODE = "full"`, re-run. Full mode is resumable and writes
   `weak_labels_v1.csv` + manifests for a Kaggle Dataset.

Engine: Qwen3-14B-AWQ (primary, ~7GB) or Qwen3-8B FP16 (fallback) (v9). vLLM dropped (CUDA 13 build on a CUDA 12 system); bitsandbytes 4-bit dropped (transformers 5.0.0's loader did not apply quantization — both 14B and 8B OOM'd in v6). 8B FP16 ~16GB fits on 2xT4.


In [ ]:
CONFIRM_VERSION = "phase1-labels-v9"  # <-- must match the clearance string in my message
NOTEBOOK_VERSION = "phase1-labels-v9"
assert CONFIRM_VERSION == NOTEBOOK_VERSION, (
    f"Version mismatch: notebook is {NOTEBOOK_VERSION}, clearance says {CONFIRM_VERSION}. "
    "Delete this file and download the current one from GitHub.")
print("Version verified:", NOTEBOOK_VERSION)


In [ ]:
# ---------------- config ----------------
MODE = "probe"  # "probe" or "full"
# v9: try Qwen3-14B-AWQ (pre-quantized 4-bit, ~7GB) first; fall back to 8B FP16.
MODEL_ID_PRIMARY = "Qwen/Qwen3-14B-AWQ"
MODEL_ID_FALLBACK = "Qwen/Qwen3-8B"
MODEL_USED = MODEL_ID_PRIMARY
OUT_DIR = "/kaggle/working/phase1_labels"
SEED = 0
PROBE_N_OTHER = 42
MAX_REPORT_TOKENS = 2500
MAX_NEW_TOKENS = 320
BATCH_SIZE = 8
GATE_MIN_MACRO_AUC = 0.86
GATE_MAX_PARSE_FAIL = 0.15
GATE_MAX_FULL_HOURS = 8.0
import os
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| OUT_DIR:", OUT_DIR)


In [ ]:
# ---------------- environment: fail fast with no GPU ----------------
import shutil, importlib, sys, subprocess, os, sysconfig, traceback
assert shutil.which("nvidia-smi") is not None, (
    "STOP: no GPU detected. Enable the GPU accelerator and re-run. "
    "Nothing has been spent.")
import torch
assert torch.cuda.is_available(), "STOP: torch sees no CUDA device."
N_GPU = torch.cuda.device_count()
print(f"GPUs: {N_GPU} x", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| torch cuda build:", torch.version.cuda)

def ensure_import(mod, pip):
    try:
        return importlib.import_module(mod)
    except ImportError:
        print(f"installing {pip} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip], check=True)
        return importlib.import_module(mod)

# Hub must be inside transformers 5.0.0's supported range (>=1.3,<2.0).
print("installing huggingface_hub>=1.3,<2.0 ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub>=1.3,<2.0"], check=True)
importlib.invalidate_caches()
import huggingface_hub
print("huggingface_hub:", huggingface_hub.__version__)

transformers = ensure_import("transformers", "transformers")
print("transformers:", transformers.__version__)
assert transformers.__version__ == "5.0.0", (
    f"transformers was upgraded to {transformers.__version__}! Re-run from a fresh session.")

# True gate: resolve the lazy quantization config attributes.
_ = transformers.FbgemmFp8Config
_ = transformers.CompressedTensorsConfig
print("transformers lazy quantization imports: OK")

# v7: NO vLLM (CUDA 13 build on CUDA 12 system; pip install poisons deps).
# v7: NO bitsandbytes (transformers 5.0.0's new loading pipeline does not
#     apply 4-bit quantization; both 14B and 8B OOM'd in v6).
# Engine is pure transformers FP16. Qwen3-8B FP16 = ~16GB, fits on 2xT4.
import pandas as pd, numpy as np
print("env OK | engine: transformers FP16 (no quantization)")


In [ ]:
# ---------------- label core (tested locally, embedded verbatim) ----------------
"""Phase 1 label-extraction core: prompt, vocabularies, parsing, calibration.

Pure-Python, no GPU deps. Tested locally; the Kaggle notebook embeds this
file verbatim (single self-contained notebook per release convention).
"""

import json
import re

# Short JSON keys -> official train.csv label names
KEY2LABEL = {
    "acl": "ACL",
    "mcl": "MCL",
    "mm": "Medial Meniscus",
    "lm": "Lateral Meniscus",
    "moa": "Medial OA",
    "loa": "Lateral OA",
    "pfoa": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers": "Baker's",
    "contusion": "Contusion",
    "fracture": "Fracture",
}
LABEL2KEY = {v: k for k, v in KEY2LABEL.items()}
KEYS = list(KEY2LABEL.keys())

# term -> rank (higher = more abnormal). "not_mentioned" is handled separately
# via the per-label prior; it is NOT part of the ordinal scale.
VOCAB = {
    "acl": {
        "finding": "anterior cruciate ligament (ACL)",
        "terms": {
            "intact": 0,
            "mucoid_degeneration": 1,
            "reconstructed": 1,
            "sprain": 2,
            "partial_tear": 3,
            "complete_tear": 4,
        },
        "notes": ("sprain = low-grade injury, fibers continuous; "
                  "complete_tear = full-thickness disruption/rupture; "
                  "reconstructed = ACL graft present (post-surgical)"),
    },
    "mcl": {
        "finding": "medial collateral ligament (MCL)",
        "terms": {
            "intact": 0,
            "sprain": 2,
            "complete_tear": 4,
        },
        "notes": "sprain covers grade 1-2 / partial injury; complete_tear = grade 3",
    },
    "mm": {
        "finding": "medial meniscus",
        "terms": {
            "intact": 0,
            "degenerative_signal": 1,
            "postoperative": 2,
            "tear": 3,
            "macerated": 4,
        },
        "notes": ("degenerative_signal = intrasubstance signal WITHOUT a morphologic tear; "
                  "tear = any morphologic tear (horizontal, flap, radial, complex); "
                  "postoperative = partial meniscectomy or repair"),
    },
    "lm": {
        "finding": "lateral meniscus",
        "terms": {
            "intact": 0,
            "degenerative_signal": 1,
            "postoperative": 2,
            "tear": 3,
            "macerated": 4,
        },
        "notes": ("degenerative_signal = intrasubstance signal WITHOUT a morphologic tear; "
                  "tear = any morphologic tear (horizontal, flap, radial, complex); "
                  "postoperative = partial meniscectomy or repair"),
    },
    "moa": {
        "finding": "medial compartment osteoarthritis / cartilage loss",
        "terms": {"none": 0, "mild": 1, "moderate": 2, "severe": 3},
        "notes": "grade the described cartilage loss / osteoarthritic change",
    },
    "loa": {
        "finding": "lateral compartment osteoarthritis / cartilage loss",
        "terms": {"none": 0, "mild": 1, "moderate": 2, "severe": 3},
        "notes": "grade the described cartilage loss / osteoarthritic change",
    },
    "pfoa": {
        "finding": "patellofemoral osteoarthritis / cartilage loss",
        "terms": {"none": 0, "mild": 1, "moderate": 2, "severe": 3},
        "notes": ("Grade the described cartilage loss / osteoarthritic change. "
                  "Includes patellar and trochlear cartilage. Multilingual: "
                  "rotula/femororrotuliano (ES), femoro-patellaire (FR)."),
    },
    "effusion": {
        "finding": "joint effusion (fluid)",
        "terms": {"none": 0, "trace_small": 1, "moderate": 2, "large": 3},
        "notes": ("'trace' or 'minimal' fluid = trace_small (present). "
                  "Multilingual: derrame (ES), epanchement (FR), Erguss (DE)."),
    },
    "synovitis": {
        "finding": "synovitis / synovial proliferation / inflamed synovium",
        "terms": {"absent": 0, "present": 3},
        "notes": ("present ONLY if the report explicitly describes synovial thickening, "
                  "synovial proliferation, synovitis, or inflamed/hypertrophic synovium. "
                  "Multilingual: sinovitis (ES), synovite (FR), Synovitis (DE). "
                  "Do NOT infer from effusion alone — effusion without synovial description = absent."),
    },
    "bakers": {
        "finding": "Baker's (popliteal) cyst",
        "terms": {"absent": 0, "present": 2, "ruptured": 3},
        "notes": "",
    },
    "contusion": {
        "finding": "bone contusion / traumatic bone marrow edema",
        "terms": {"absent": 0, "present": 3},
        "notes": ("Multilingual: contusion (ES/FR), Kontusion (DE). "
                  "Bone bruise, marrow edema from trauma. NOT degenerative edema."),
    },
    "fracture": {
        "finding": "fracture (cortical break / fracture line)",
        "terms": {"absent": 0, "present": 4},
        "notes": ("Multilingual: fractura (ES), fracture (FR), Fraktur (DE). "
                  "Includes occult, stress, insufficiency, and avulsion fractures if described."),
    },
}

# Formatting-only aliases applied AFTER lower/strip/underscore normalization.
# Only unambiguous radiology synonyms; anything else is a model error.
ALIASES = {
    "acl": {
        "full_tear": "complete_tear", "full_thickness_tear": "complete_tear",
        "rupture": "complete_tear", "ruptured": "complete_tear",
        "partial_thickness_tear": "partial_tear",
        "graft": "reconstructed", "acl_graft": "reconstructed",
        "normal": "intact", "no_abnormality": "intact",
    },
    "mcl": {
        "rupture": "complete_tear", "ruptured": "complete_tear",
        "grade_3": "complete_tear", "grade_1": "sprain", "grade_2": "sprain",
        "normal": "intact",
    },
    "mm": {"rupture": "tear", "torn": "tear", "normal": "intact"},
    "lm": {"rupture": "tear", "torn": "tear", "normal": "intact"},
    "moa": {"normal": "none", "no_oa": "none"},
    "loa": {"normal": "none", "no_oa": "none"},
    "pfoa": {"normal": "none", "no_oa": "none"},
    "effusion": {"trace": "trace_small", "minimal": "trace_small", "small": "trace_small",
                 "no_effusion": "none"},
    "synovitis": {"no_synovitis": "absent"},
    "bakers": {"no_cyst": "absent", "cyst": "present"},
    "contusion": {"bone_marrow_edema": "present", "bme": "present"},
    "fracture": {"avulsion": "present", "stress_fracture": "present",
                 "insufficiency_fracture": "present"},
}
GLOBAL_ALIASES = {"unmentioned": "not_mentioned", "not_stated": "not_mentioned",
                  "n_a": "not_mentioned", "na": "not_mentioned"}


def build_user_content(report: str) -> str:
    lines = []
    for k in KEYS:
        v = VOCAB[k]
        terms = " | ".join(v["terms"].keys())
        lines.append(f"- {k} ({v['finding']}): {terms} | not_mentioned")
        if v["notes"]:
            lines.append(f"  ({v['notes']})")
    vocab_block = "\n".join(lines)
    return f"""You are a musculoskeletal radiology reader. Read the knee MRI report below. It may be written in Spanish, French, English, German, or another language — read it regardless of language.

For each of the 12 findings, choose the SINGLE allowed term that best matches what the report STATES. Rules:
- If the report does not mention the finding at all, use "not_mentioned".
- Do NOT infer findings that are not stated.
- Negated findings ("no tear", "sin rotura", "pas de déchirure", "kein Riss") map to the intact/absent/none term.
- Uncertain/hedged language ("possible", "cannot exclude", "suggestive of") still maps to the matching positive term — choose the term, not the hedge.

Findings and allowed terms:
{vocab_block}

Your ENTIRE response must be exactly the JSON object and nothing else. Do not add any text before or after. Return ONLY a JSON object with exactly these 12 keys, in this order:
{json.dumps(KEYS)}
Example: {{"acl": "intact", "mcl": "not_mentioned", ...}}
No explanations. No markdown. No extra text.

Report:
\"\"\"{report}\"\"\""""


def build_prompt(report: str) -> str:
    return build_user_content(report)


def _strip_think(text: str) -> str:
    # Defensive: Qwen3 thinking traces must never reach the JSON parser.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    return text


def extract_json(text: str):
    """Return parsed dict or None. Finds the first {...} span."""
    text = _strip_think(text)
    m = re.search(r"\{", text)
    if not m:
        return None
    # balanced-brace scan from first '{'
    depth = 0
    start = m.start()
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    obj = json.loads(text[start:i + 1])
                except Exception:
                    return None
                return obj if isinstance(obj, dict) else None
    return None


def _canon_term(key: str, raw: str):
    t = raw.strip().lower().replace("-", "_").replace(" ", "_").replace("/", "_")
    t = re.sub(r"_+", "_", t).strip("_")
    if t in GLOBAL_ALIASES:
        t = GLOBAL_ALIASES[t]
    if t in VOCAB[key]["terms"] or t == "not_mentioned":
        return t
    alias = ALIASES.get(key, {}).get(t)
    if alias:
        return alias
    return None


def normalize_descriptors(raw: dict):
    """-> (terms dict key->term|'not_mentioned', issues list)."""
    terms, issues = {}, []
    for k in KEYS:
        if k not in raw:
            terms[k] = "not_mentioned"
            issues.append(f"missing key: {k}")
            continue
        v = raw[k]
        if not isinstance(v, str):
            terms[k] = "not_mentioned"
            issues.append(f"non-string value for {k}: {v!r}")
            continue
        c = _canon_term(k, v)
        if c is None:
            terms[k] = "not_mentioned"
            issues.append(f"unknown term for {k}: {v!r}")
        else:
            terms[k] = c
    extra = [k for k in raw.keys() if k not in KEYS]
    if extra:
        issues.append(f"extra keys ignored: {extra}")
    return terms, issues


def rank_of(key: str, term: str):
    """Ordinal rank, or None for not_mentioned."""
    if term == "not_mentioned":
        return None
    return VOCAB[key]["terms"][term]


# ---- isotonic calibration (PAVA), no sklearn needed ----

def _pava_fit(xs, ys):
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    xs = [xs[i] for i in order]
    ys = [float(ys[i]) for i in order]
    sums, counts, xmins = [], [], []
    for x, y in zip(xs, ys):
        sums.append(y)
        counts.append(1)
        xmins.append(x)
        while (len(sums) >= 2
               and sums[-2] / counts[-2] > sums[-1] / counts[-1] + 1e-12):
            sums[-2] += sums[-1]
            counts[-2] += counts[-1]
            sums.pop()
            counts.pop()
            xmins.pop()
    # merge blocks sharing the same x (isotonic constraint is vacuous there)
    merged = []
    for xm, s, c in zip(xmins, sums, counts):
        if merged and merged[-1][0] == xm:
            px, ps, pc = merged[-1]
            merged[-1] = (px, ps + s, pc + c)
        else:
            merged.append((xm, s, c))
    return [(xm, s / c) for xm, s, c in merged]


def _pava_predict(levels, x):
    v = levels[0][1]
    for xmin, val in levels:
        if x >= xmin:
            v = val
        else:
            break
    return v


def calibrate_label(ranks, golds, prior):
    """Fit per-label calibration.

    ranks: list of float|None (None = not_mentioned); golds: 0/1 ints.
    Returns dict(prior=..., levels=[(rank, score)...] or None, n_fit=...).
    """
    pairs = [(r, g) for r, g in zip(ranks, golds) if r is not None]
    if len(pairs) < 4 or len({g for _, g in pairs}) < 2:
        return {"prior": prior, "levels": None, "n_fit": len(pairs)}
    xs = [p[0] for p in pairs]
    ys = [p[1] for p in pairs]
    levels = _pava_fit(xs, ys)
    return {"prior": prior, "levels": levels, "n_fit": len(pairs)}


def apply_calibration(key, term, cal):
    if term == "not_mentioned":
        return cal["prior"]
    r = rank_of(key, term)
    if cal["levels"] is None:
        # fallback: linear in rank fraction (only when too few gold mentions)
        maxr = max(VOCAB[key]["terms"].values())
        return 0.05 + 0.90 * (r / maxr)
    return float(_pava_predict(cal["levels"], r))


def rank_score(key, term):
    """Uncalibrated ordinal score in [0,1] — same AUC as calibrated."""
    if term == "not_mentioned":
        return 0.0
    maxr = max(VOCAB[key]["terms"].values())
    return rank_of(key, term) / maxr


# ---- metrics ----

def auc_score(y_true, y_score):
    """Binary AUC via Mann-Whitney. Returns None if degenerate."""
    y_true = list(y_true)
    y_score = list(y_score)
    n1 = sum(y_true)
    n0 = len(y_true) - n1
    if n1 == 0 or n0 == 0:
        return None
    order = sorted(range(len(y_score)), key=lambda i: y_score[i])
    ranks = [0.0] * len(y_score)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and y_score[order[j + 1]] == y_score[order[i]]:
            j += 1
        avg = (i + j) / 2.0 + 1  # 1-based average rank
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    s = sum(ranks[i] for i in range(len(y_true)) if y_true[i] == 1)
    return (s - n1 * (n1 + 1) / 2.0) / (n1 * n0)


def macro_auc(gold: dict, scores: dict):
    """gold/scores: label -> list. Returns (macro, {label: (auc, n_pos, n)})."""
    per = {}
    vals = []
    for lab in gold:
        a = auc_score(gold[lab], scores[lab])
        n1 = sum(gold[lab])
        per[lab] = (a, n1, len(gold[lab]))
        if a is not None:
            vals.append(a)
    macro = sum(vals) / len(vals) if vals else None
    return macro, per


In [ ]:
# ---------------- engine (transformers FP16, Qwen3-8B) ----------------
ENGINE = None
_tokenizer = None
_llm = None

def _apply_template(report_text):
    """Chat-format with thinking disabled (Qwen3). Falls back gracefully."""
    msgs = [{"role": "user", "content": build_user_content(report_text)}]
    try:
        return _tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return _tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

def _truncate(report_text):
    ids = _tokenizer.encode(report_text, add_special_tokens=False)
    if len(ids) > MAX_REPORT_TOKENS:
        _truncate.n_truncated += 1
        ids = ids[:MAX_REPORT_TOKENS]
    return _tokenizer.decode(ids)
_truncate.n_truncated = 0

def load_engine():
    global ENGINE, _tokenizer, _llm, MODEL_USED
    from transformers import AutoModelForCausalLM
    # Try 14B-AWQ first (pre-quantized, different loader than broken bnb).
    for model_id, kwargs in [
        (MODEL_ID_PRIMARY, dict(device_map={"": 0}, trust_remote_code=True)),
        (MODEL_ID_FALLBACK, dict(torch_dtype=torch.float16, device_map="auto",
                                 max_memory={0: "13GiB", 1: "13GiB"},
                                 trust_remote_code=True)),
    ]:
        try:
            print("loading", model_id, "...")
            _tokenizer = transformers.AutoTokenizer.from_pretrained(
                model_id, trust_remote_code=True)
            _llm = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
            _llm.eval()
            MODEL_USED = model_id
            ENGINE = "hf-awq" if "AWQ" in model_id else "hf-fp16"
            try:
                fp = _llm.get_memory_footprint() / 1e9
                print(f"engine: {ENGINE} + {MODEL_USED} ({fp:.1f} GB footprint)")
            except Exception:
                print("engine:", ENGINE, "+", MODEL_USED)
            return
        except Exception as e:
            print(f"{model_id} failed: {type(e).__name__}: {str(e)[:200]}")
            torch.cuda.empty_cache()
    raise RuntimeError("Both 14B-AWQ and 8B-FP16 failed to load.")

def generate(texts):
    """texts: list of formatted prompts -> list of raw output strings."""
    out_texts = []
    BS = BATCH_SIZE
    for i in range(0, len(texts), BS):
        batch = texts[i:i + BS]
        enc = _tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                         max_length=8192 - MAX_NEW_TOKENS).to(_llm.device)
        with torch.no_grad():
            gen = _llm.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                pad_token_id=_tokenizer.eos_token_id)
        for b in range(len(batch)):
            gen_ids = gen[b][enc["input_ids"].shape[1]:]
            out_texts.append(_tokenizer.decode(gen_ids, skip_special_tokens=True))
    return out_texts


In [ ]:
# ---------------- data ----------------
import glob, hashlib
cands = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert cands, "train.csv not found under /kaggle/input — attach the competition data."
TRAIN_CSV = cands[0]
print("train.csv:", TRAIN_CSV)
# latin-1: pandas' default silently mangles accented chars in Report
df = pd.read_csv(TRAIN_CSV, encoding="latin-1")
LABELS = [KEY2LABEL[k] for k in KEYS]
assert all(c in df.columns for c in LABELS), "label columns mismatch"
assert "StudyInstanceUID" in df.columns and "Report" in df.columns
print("studies:", len(df))

gold_mask = df[LABELS].notna().all(axis=1)
n_gold = int(gold_mask.sum())
print("gold studies (all 12 labels):", n_gold)
assert n_gold >= 40, f"only {n_gold} gold studies — CSV read looks wrong, stopping."
N_EMPTY = int(df["Report"].fillna("").str.strip().eq("").sum())
print("empty reports:", N_EMPTY)

df["_report"] = df["Report"].fillna("").astype(str)
df["_hash"] = df["_report"].apply(lambda t: hashlib.sha1(t.encode("utf-8", "replace")).hexdigest())
gold_df = df[gold_mask].reset_index(drop=True)
uniq = df.drop_duplicates("_hash").reset_index(drop=True)
print(f"unique reports: {len(uniq)} (from {len(df)} studies)")
hash2uids = df.groupby("_hash")["StudyInstanceUID"].apply(list).to_dict()
gold_hashes = sorted(gold_df["_hash"].unique().tolist())
print("unique gold reports:", len(gold_hashes))

rng = np.random.RandomState(SEED)
other_hashes = sorted(set(uniq["_hash"]) - set(gold_hashes))
probe_extra = sorted(rng.choice(other_hashes, size=min(PROBE_N_OTHER, len(other_hashes)),
                                replace=False).tolist())
print("probe extra reports:", len(probe_extra))
hash2report = dict(zip(uniq["_hash"], uniq["_report"]))

In [ ]:
# ---------------- extraction ----------------
import time, json

def extract_terms(report_hashes):
    """Run the engine over report hashes -> list of record dicts (with one retry)."""
    reports = [_truncate(hash2report[h]) for h in report_hashes]
    prompts = [_apply_template(r) for r in reports]
    print(f"truncated { _truncate.n_truncated } long reports at {MAX_REPORT_TOKENS} tokens")
    t0 = time.time()
    raws = generate(prompts)
    dt = time.time() - t0
    print(f"generated {len(raws)} in {dt:.1f}s ({len(raws)/dt:.2f} reports/s)")

    records = []
    n_retry = 0
    need_retry_idx = []
    for h, raw in zip(report_hashes, raws):
        obj = extract_json(raw)
        if obj is None:
            need_retry_idx.append(len(records))
            records.append((h, raw, None, ["unparseable output"]))
            continue
        terms, issues = normalize_descriptors(obj)
        rec = {"report_hash": h, "study_uids": hash2uids[h], "terms": terms,
               "issues": issues, "parse_ok": len(issues) == 0,
               "raw_preview": raw[:200]}
        records.append((h, raw, rec, issues))
        if issues:
            need_retry_idx.append(len(records) - 1)
    # one repair retry for anything with issues
    if need_retry_idx:
        rprompts = []
        for i in need_retry_idx:
            h, raw, rec, issues = records[i]
            fix = ("Your previous response had these problems:\n- " + "\n- ".join(issues) +
                   "\nReturn ONLY the corrected JSON object with exactly the 12 keys " +
                   json.dumps(KEYS) + " and only allowed terms. No other text.")
            rprompts.append(_apply_template(hash2report[h]) + "\n\n" + fix)
        rraws = generate(rprompts)
        n_retry = len(rraws)
        for i, rraw in zip(need_retry_idx, rraws):
            h, raw, rec, issues = records[i]
            obj = extract_json(rraw)
            if obj is not None:
                terms2, issues2 = normalize_descriptors(obj)
            else:
                terms2 = {k: "not_mentioned" for k in KEYS}
                issues2 = ["unparseable output after retry"]
            records[i] = (h, rraw, {"report_hash": h, "study_uids": hash2uids[h],
                                    "terms": terms2, "issues": issues2,
                                    "parse_ok": len(issues2) == 0,
                                    "raw_preview": rraw[:200], "retried": True}, issues2)
    final = [r[2] for r in records]
    n_fail = sum(1 for r in final if not r["parse_ok"])
    print(f"parse failures after retry: {n_fail}/{len(final)} ({n_fail/max(1,len(final)):.1%}); retries: {n_retry}")
    for r in final:
        r["engine"] = ENGINE
        r["model"] = MODEL_USED
    return final, dt

load_engine()
val_hashes = gold_hashes + (probe_extra if MODE == "probe" else [])
val_records, val_dt = extract_terms(val_hashes)
rec_by_hash = {r["report_hash"]: r for r in val_records}

In [ ]:
# ---------------- calibration + validation vs gold ----------------
def study_terms(uid):
    h = df.loc[df["StudyInstanceUID"] == uid, "_hash"].iloc[0]
    return rec_by_hash[h]["terms"]

gold_scores, gold_true = {}, {}
for k in KEYS:
    lab = KEY2LABEL[k]
    ranks, trues = [], []
    for _, row in gold_df.iterrows():
        t = study_terms(row["StudyInstanceUID"])
        ranks.append(rank_of(k, t[k]))
        trues.append(int(row[lab]))
    prior = sum(trues) / len(trues)
    cal = calibrate_label(ranks, trues, prior)
    gold_scores[lab] = [apply_calibration(k, t, cal)
                        for t in (study_terms(u)[k] for u in gold_df["StudyInstanceUID"])]
    gold_true[lab] = trues
    globals()[f"CAL_{k}"] = cal

macro, per = macro_auc(gold_true, gold_scores)
print(f"macro AUC vs gold ({len(gold_df)} studies): {macro:.4f}" if macro else "macro AUC: undefined")
for lab in LABELS:
    a, n1, n = per[lab]
    print(f"  {lab:16s} AUC={a if a is None else f'{a:.3f}'}  positives={n1}/{n}")

n_val = len(val_records)
parse_fail = sum(1 for r in val_records if not r["parse_ok"]) / n_val
per_report_s = val_dt / n_val
n_unique_full = len(uniq)
proj_h = per_report_s * n_unique_full / 3600
print(f"\nparse-fail rate: {parse_fail:.1%} | per-report: {per_report_s:.2f}s | "
      f"projected full run ({n_unique_full} reports): {proj_h:.2f}h")

gates = {
    "macro_auc>=0.86": macro is not None and macro >= GATE_MIN_MACRO_AUC,
    "parse_fail<15%": parse_fail < GATE_MAX_PARSE_FAIL,
    "projected<8h": proj_h < GATE_MAX_FULL_HOURS,
}
print("\nGATES:", gates)
assert all(gates.values()), (
    "GATE FAILED — do not run full mode. Inspect per-label AUC above; likely fixes: "
    "prompt wording for the weak labels, or (if slow) the vLLM path.")
print("PROBE PASSED — full mode is cleared.") if MODE == "probe" else None

In [ ]:
# ---------------- full mode: bulk extraction + outputs ----------------
if MODE != "full":
    print("probe mode: stopping here. Flip MODE='full' and re-run only after PROBE PASSED.")
else:
    import json as _json
    jsonl_path = os.path.join(OUT_DIR, "descriptors.jsonl")
    done = set()
    if os.path.exists(jsonl_path):
        with open(jsonl_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    done.add(_json.loads(line)["report_hash"])
    todo = [h for h in sorted(set(uniq["_hash"])) if h not in done and h not in rec_by_hash]
    print(f"resuming: {len(done)} done, {len(todo)} to do (+{len(rec_by_hash)} validated)")

    BS = 64
    with open(jsonl_path, "a") as f:
        # persist the already-validated records first (once)
        if not done:
            for r in val_records:
                f.write(_json.dumps(r) + "\n")
            f.flush()
            done.update(rec_by_hash.keys())
        for i in range(0, len(todo), BS):
            batch = todo[i:i + BS]
            recs, _ = extract_terms(batch)
            for r in recs:
                f.write(_json.dumps(r) + "\n")
            f.flush()
            print(f"[{min(i+BS, len(todo))}/{len(todo)}] flushed", flush=True)

    # reload everything, apply calibration, write CSV
    all_recs = {}
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                r = _json.loads(line)
                all_recs[r["report_hash"]] = r
    assert len(all_recs) == len(uniq), f"{len(all_recs)} vs {len(uniq)} unique reports"
    cals = {k: globals()[f"CAL_{k}"] for k in KEYS}
    with open(os.path.join(OUT_DIR, "calibration.json"), "w") as f:
        _json.dump({KEY2LABEL[k]: {"prior": c["prior"], "levels": c["levels"], "n_fit": c["n_fit"]}
                                  for k, c in cals.items()}, f, indent=1)

    rows = []
    for _, srow in df.iterrows():
        terms = all_recs[srow["_hash"]]["terms"]
        rows.append([srow["StudyInstanceUID"]] +
                    [apply_calibration(k, terms[k], cals[k]) for k in KEYS])
    wdf = pd.DataFrame(rows, columns=["StudyInstanceUID"] + LABELS)
    csv_path = os.path.join(OUT_DIR, "weak_labels_v1.csv")
    wdf.to_csv(csv_path, index=False)

    # verify the artifact we just wrote
    chk = pd.read_csv(csv_path)
    assert len(chk) == len(df) and list(chk.columns) == ["StudyInstanceUID"] + LABELS
    assert chk[LABELS].notna().all().all(), "NaN scores found"
    assert ((chk[LABELS] >= 0) & (chk[LABELS] <= 1)).all().all(), "scores out of [0,1]"
    manifest = {
        "phase": "phase1-weak-labels-v1",
        "notebook_version": NOTEBOOK_VERSION,
        "model": MODEL_USED,
        "engine": ENGINE,
        "n_studies": len(df), "n_unique_reports": len(uniq),
        "n_gold": len(gold_df), "n_empty_reports": N_EMPTY,
        "gold_macro_auc": macro,
        "gold_per_label_auc": {lab: per[lab][0] for lab in LABELS},
        "parse_fail_rate_val": parse_fail,
    }
    with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
        _json.dump(manifest, f, indent=1)
    print("wrote:", csv_path, f"({os.path.getsize(csv_path)/1e6:.1f} MB)")
    print("manifest:", manifest)
    print("""dataset-metadata.json template (replace YOUR_KAGGLE_USERNAME):
{
 "title": "rsna-knee-phase1-weak-labels-v1",
 "id": "YOUR_KAGGLE_USERNAME/rsna-knee-phase1-weak-labels-v1",
 "licenses": [{"name": "CC0-1.0"}]
}
Files to publish: weak_labels_v1.csv, descriptors.jsonl, calibration.json, manifest.json""")
    print("\nPHASE 1 COMPLETE — weak labels are extracted, calibrated, and versioned.")